In [ ]:
import sys
import subprocess

print("Installing required Python libraries...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "gspread",
    "google-auth",
    "openai",
    "python-dotenv"
])

print("Required Python libraries installed successfully.")

In [ ]:
import os
import json
import gspread
import google.auth

from decimal import Decimal, ROUND_HALF_UP
from openai import OpenAI
from dotenv import load_dotenv

print("Required libraries imported successfully.")

In [ ]:
import os
import importlib
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    try:
        userdata = importlib.import_module("google.colab.userdata")
        OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    except:
        OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. "
        "Add it to the .env file in VS Code "
        "or to Colab Secrets."
    )

print("OpenRouter API key loaded successfully.")

In [ ]:
import os
import glob
import gspread
import importlib

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

try:
    colab_auth = importlib.import_module("google.colab.auth")
    google_auth = importlib.import_module("google.auth")

    colab_auth.authenticate_user()

    creds, _ = google_auth.default(
        scopes=SCOPES
    )

    print("Google Colab authentication completed successfully.")

except ModuleNotFoundError:
    from google_auth_oauthlib.flow import InstalledAppFlow

    credentials_files = glob.glob(
        os.path.join("credentials", "*.json")
    )

    credentials_files = [
        file for file in credentials_files
        if not os.path.basename(file).startswith("token")
    ]

    if not credentials_files:
        raise FileNotFoundError(
            "No Google OAuth JSON file found in the credentials folder."
        )

    if len(credentials_files) > 1:
        raise ValueError(
            "Multiple Google OAuth JSON files found. "
            "Keep only one credentials JSON file in the credentials folder."
        )

    credentials_file = credentials_files[0]

    flow = InstalledAppFlow.from_client_secrets_file(
        credentials_file,
        SCOPES
    )

    creds = flow.run_local_server(port=0)

    print("Google Sheets authentication completed successfully.")

sheets_client = gspread.authorize(creds)

print("Google Sheets connection established successfully.")

In [ ]:
GOOGLE_SHEET_ID = "1QTHrfg9dMz2KeG29jYTwD0Vo0iJrr13f25CK8jSZ4J8"

if GOOGLE_SHEET_ID == "YOUR_GOOGLE_SHEET_ID":
    raise ValueError(
        "Please enter your Google Sheet ID in GOOGLE_SHEET_ID."
    )

spreadsheet = sheets_client.open_by_key(
    GOOGLE_SHEET_ID
)

payroll_sheet = spreadsheet.worksheet("Payroll")
attendance_sheet = spreadsheet.worksheet("Attendance")
sales_sheet = spreadsheet.worksheet("Sales")
attendance_policy_sheet = spreadsheet.worksheet("Attendance_Policy")
commission_policy_sheet = spreadsheet.worksheet("Commission_Policy")

print("Google Sheets workbook connected successfully.")
print("Required worksheets connected successfully.")

In [ ]:
payroll_data = payroll_sheet.get_all_records()
attendance_data = attendance_sheet.get_all_records()
sales_data = sales_sheet.get_all_records()
attendance_policy_data = attendance_policy_sheet.get_all_records()
commission_policy_data = commission_policy_sheet.get_all_records()

print("Payroll data loaded successfully.")
print("Attendance data loaded successfully.")
print("Sales data loaded successfully.")
print("Attendance policy data loaded successfully.")
print("Commission policy data loaded successfully.")

In [ ]:
from decimal import Decimal, ROUND_HALF_UP


def number(value):
    try:
        return float(value)
    except:
        return 0.0


def money(value):
    return int(
        Decimal(str(value)).quantize(
            Decimal("1"),
            rounding=ROUND_HALF_UP
        )
    )


def percentage(value):
    try:
        return float(
            str(value).replace("%", "").strip()
        )
    except:
        return 0.0


def records(sheet):
    return sheet.get_all_records()


def get_attendance(employee_id):
    return [
        row for row in records(attendance_sheet)
        if str(row.get("Employee ID", "")).strip() == employee_id
    ]


def get_sales(employee_id):
    return [
        row for row in records(sales_sheet)
        if str(row.get("Employee ID", "")).strip() == employee_id
    ]


def get_attendance_policy():
    return records(attendance_policy_sheet)


def get_commission_policy():
    return records(commission_policy_sheet)


def get_payroll(employee_id):
    rows = [
        row for row in records(payroll_sheet)
        if str(row.get("Employee ID", "")).strip() == employee_id
    ]

    if not rows:
        raise ValueError(
            f"No payroll record found for employee {employee_id}."
        )

    return rows[0]


def active_policy(rows):
    active_rows = []

    for row in rows:
        active = str(
            row.get("Active", "")
        ).strip().lower()

        effective_from = str(
            row.get("Effective From", "")
        ).strip()

        if active == "yes" and effective_from:
            active_rows.append(row)

    return active_rows


def attendance_pay(employee_id):
    attendance = get_attendance(employee_id)
    policies = active_policy(
        get_attendance_policy()
    )

    late_rate = 0
    early_rate = 0

    overtime_policies = {}

    for policy in policies:
        policy_type = str(
            policy.get("Policy Type", "")
        ).strip()

        day_type = str(
            policy.get("Day Type", "")
        ).strip()

        if (
            policy_type == "Late Entry"
            and day_type == "Working Day"
        ):
            late_rate = number(
                policy.get("Hourly Rate", 0)
            )

        elif (
            policy_type == "Early Exit"
            and day_type == "Working Day"
        ):
            early_rate = number(
                policy.get("Hourly Rate", 0)
            )

        elif policy_type == "Overtime":
            overtime_policies[day_type] = {
                "minimum": number(
                    policy.get(
                        "Daily Minimum Payable Hours",
                        0
                    )
                ),
                "maximum": number(
                    policy.get(
                        "Daily Maximum Payable Hours",
                        0
                    )
                ),
                "rate": number(
                    policy.get("Hourly Rate", 0)
                )
            }

    late_deduction = 0
    early_deduction = 0
    overtime_pay = 0

    overtime_violations = []

    for row in attendance:
        day_type = str(
            row.get("Day Type", "")
        ).strip()

        late_hours = number(
            row.get("Late Entry Hours", 0)
        )

        early_hours = number(
            row.get("Early Exit Hours", 0)
        )

        overtime_hours = number(
            row.get("Overtime Hours", 0)
        )

        late_deduction += (
            late_hours * late_rate
        )

        early_deduction += (
            early_hours * early_rate
        )

        if (
            overtime_hours > 0
            and day_type in overtime_policies
        ):
            policy = overtime_policies[
                day_type
            ]

            minimum_hours = policy[
                "minimum"
            ]

            maximum_hours = policy[
                "maximum"
            ]

            hourly_rate = policy[
                "rate"
            ]

            if overtime_hours >= minimum_hours:
                payable_hours = min(
                    overtime_hours,
                    maximum_hours
                )

                overtime_pay += (
                    payable_hours *
                    hourly_rate
                )

            else:
                payable_hours = 0

            if overtime_hours > maximum_hours:
                overtime_violations.append({
                    "date": row.get(
                        "Attendance Date",
                        ""
                    ),
                    "day_type": day_type,
                    "actual_hours": overtime_hours,
                    "payable_hours": maximum_hours,
                    "excess_hours": (
                        overtime_hours -
                        maximum_hours
                    )
                })

    return {
        "late_deduction": money(
            late_deduction
        ),
        "early_deduction": money(
            early_deduction
        ),
        "overtime_pay": money(
            overtime_pay
        ),
        "overtime_violations": (
            overtime_violations
        )
    }


def commission_pay(employee_id):
    sales = get_sales(employee_id)
    policies = active_policy(
        get_commission_policy()
    )

    total_sales = sum(
        number(
            row.get("Sales Amount", 0)
        )
        for row in sales
    )

    applicable_rate = 0

    for policy in policies:
        minimum = number(
            policy.get(
                "From Sales Amount",
                0
            )
        )

        maximum = number(
            policy.get(
                "To Sales Amount",
                0
            )
        )

        rate = percentage(
            policy.get(
                "Commission Percent",
                0
            )
        )

        if minimum <= total_sales <= maximum:
            applicable_rate = rate
            break

    commission = (
        total_sales *
        applicable_rate /
        100
    )

    return {
        "sales": total_sales,
        "rate": applicable_rate,
        "commission": money(
            commission
        )
    }


def verify_payroll(employee_id):
    payroll = get_payroll(
        employee_id
    )

    commission = commission_pay(
        employee_id
    )

    attendance = attendance_pay(
        employee_id
    )

    expected = {
        "commission": commission[
            "commission"
        ],
        "overtime": attendance[
            "overtime_pay"
        ],
        "total_earnings": money(
            commission["commission"] +
            attendance["overtime_pay"]
        ),
        "late_deduction": attendance[
            "late_deduction"
        ],
        "early_deduction": attendance[
            "early_deduction"
        ],
        "total_deductions": money(
            attendance["late_deduction"] +
            attendance["early_deduction"]
        )
    }

    current = {
        "commission": money(
            payroll.get("Commission", 0)
        ),
        "overtime": money(
            payroll.get("Overtime", 0)
        ),
        "total_earnings": money(
            payroll.get("Total Earnings", 0)
        ),
        "late_deduction": money(
            payroll.get("Late Deduction", 0)
        ),
        "early_deduction": money(
            payroll.get("Early Deduction", 0)
        ),
        "total_deductions": money(
            payroll.get("Total Deductions", 0)
        )
    }

    differences = {
        key: current[key] - expected[key]
        for key in expected
    }

    earnings_correct = (
        differences["commission"] == 0
        and differences["overtime"] == 0
        and differences["total_earnings"] == 0
    )

    deductions_correct = (
        differences["late_deduction"] == 0
        and differences["early_deduction"] == 0
        and differences["total_deductions"] == 0
    )

    return {
        "employee_id": employee_id,
        "employee": payroll.get(
            "Employee", ""
        ),
        "salary_slip_id": payroll.get(
            "Salary Slip ID", ""
        ),
        "currency": payroll.get(
            "Currency", "Rs"
        ),
        "payroll": current,
        "expected": expected,
        "differences": differences,
        "commission_sales": commission[
            "sales"
        ],
        "commission_rate": commission[
            "rate"
        ],
        "overtime_violations": (
            attendance[
                "overtime_violations"
            ]
        ),
        "earnings_correct": (
            earnings_correct
        ),
        "deductions_correct": (
            deductions_correct
        ),
        "discrepancies": not (
            earnings_correct
            and deductions_correct
        )
    }


def update_payroll(
    employee_id,
    approved=None
):
    verification = verify_payroll(
        employee_id
    )

    if not verification[
        "discrepancies"
    ]:
        return {
            "status": "No update required",
            "employee_id": employee_id,
            "message": (
                "Payroll already matches "
                "the expected values."
            )
        }

    if approved is None:
        return {
            "status": "Approval required",
            "employee_id": employee_id,
            "differences": (
                verification[
                    "differences"
                ]
            )
        }

    if approved is False:
        return {
            "status": "Update blocked",
            "employee_id": employee_id,
            "message": (
                "Human approval was not provided."
            )
        }

    row_number = None

    payroll_rows = records(
        payroll_sheet
    )

    for index, row in enumerate(
        payroll_rows,
        start=2
    ):
        if str(
            row.get("Employee ID", "")
        ).strip() == employee_id:
            row_number = index
            break

    if row_number is None:
        raise ValueError(
            f"No payroll row found for employee {employee_id}."
        )

    expected = verification[
        "expected"
    ]

    payroll_sheet.update(
        range_name=f"G{row_number}:L{row_number}",
        values=[[
            expected["commission"],
            expected["overtime"],
            expected["total_earnings"],
            expected["late_deduction"],
            expected["early_deduction"],
            expected["total_deductions"]
        ]]
    )

    return {
        "status": "Updated",
        "employee_id": employee_id
    }


print("Payroll verification functions and agent tools are ready.")

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

tools = {
    "verify_payroll": verify_payroll
}

tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "verify_payroll",
            "description": (
                "Verify an employee payroll record against "
                "active commission and attendance policies. "
                "Python calculations are authoritative."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string",
                        "description": (
                            "Employee ID to verify"
                        )
                    }
                },
                "required": [
                    "employee_id"
                ]
            }
        }
    }
]

system_prompt = """
You are an AI Payroll Verification Agent.

Your role is to coordinate payroll verification tools
and explain their results.

Python calculations and verification results are
authoritative.

Do not recalculate, modify, round, or invent payroll
numbers.

Use the verify_payroll tool to obtain the payroll
verification result.

After receiving the verification result:

1. Explain whether earnings are correct.
2. Explain whether deductions are correct.
3. Identify any payroll discrepancies.
4. Identify any overtime policy violations.
5. Explain what should be reviewed.

If there are no payroll discrepancies, do not request
human approval and do not request a payroll update.

If payroll discrepancies exist, human approval is
required before any update.

The update_payroll function is controlled by the Python
workflow and is not available as an LLM tool.

Keep numerical values exactly as provided by the
verification tool.
"""

print("Agent configuration and verification tool registry are ready.")

In [ ]:
def format_python_result(result):
    explanation = []

    if result["earnings_correct"]:
        explanation.append(
            "1. Earnings are correct; calculated earnings match expected."
        )
    else:
        explanation.append(
            "1. Earnings have discrepancies and require review."
        )

    if result["deductions_correct"]:
        explanation.append(
            "2. Deductions are correct; calculated deductions match expected."
        )
    else:
        explanation.append(
            "2. Deductions have discrepancies and require review."
        )

    if result["discrepancies"]:
        explanation.append(
            "3. Payroll discrepancies were found."
        )
    else:
        explanation.append(
            "3. No payroll discrepancies were found."
        )

    if result["overtime_violations"]:
        explanation.append(
            f"4. Overtime policy violation(s) found: "
            f"{len(result['overtime_violations'])}."
        )
    else:
        explanation.append(
            "4. No overtime policy violations were found."
        )

    if result["overtime_violations"]:
        explanation.append(
            "5. Review the overtime violation(s) before payroll approval."
        )
    elif result["discrepancies"]:
        explanation.append(
            "5. Review the payroll discrepancies before approval."
        )
    else:
        explanation.append(
            "5. No payroll approval is required because the payroll "
            "already matches the expected values."
        )

    return "\n".join(explanation)


def verify_and_approve(employee_id):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"Verify payroll for employee {employee_id}."
            )
        }
    ]

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=messages,
        tools=tool_schemas,
        tool_choice="auto"
    )

    message = response.choices[0].message
    verification_result = None

    while message.tool_calls:
        messages.append(
            message.model_dump()
        )

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name

            tool_args = json.loads(
                tool_call.function.arguments
            )

            if tool_name not in tools:
                raise ValueError(
                    f"Unauthorized tool requested: {tool_name}"
                )

            tool_result = tools[tool_name](
                **tool_args
            )

            if tool_name == "verify_payroll":
                verification_result = tool_result

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(tool_result)
            })

        response = client.chat.completions.create(
            model="openrouter/free",
            messages=messages,
            tools=tool_schemas,
            tool_choice="auto"
        )

        message = response.choices[0].message

    if verification_result is None:
        verification_result = verify_payroll(
            employee_id
        )

    explanation_response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "system",
                "content": (
                    "You explain payroll verification results. "
                    "Python is authoritative. "
                    "Never recalculate, modify, round, or invent numbers."
                )
            },
            {
                "role": "user",
                "content": (
                    "Explain the verification result concisely. "
                    "Use only facts contained in the result. "
                    "Do not calculate or change any numbers.\n\n"
                    f"{json.dumps(verification_result, indent=2)}"
                )
            }
        ]
    )

    print("Agent Explanation")
    print("-----------------")
    print(
        explanation_response.choices[0].message.content
    )

    print()
    print("Python Verification Result")
    print("--------------------------")
    print(
        format_python_result(
            verification_result
        )
    )

    print()
    print("Payroll Verification")
    print("--------------------")
    print(
        f"Employee: {verification_result['employee']}"
    )
    print(
        f"Employee ID: {verification_result['employee_id']}"
    )
    print(
        f"Salary Slip: {verification_result['salary_slip_id']}"
    )
    print(
        f"Currency: {verification_result['currency']}"
    )

    print()
    print("Current Payroll:")

    for key, value in verification_result[
        "payroll"
    ].items():
        print(
            f"{key}: {value}"
        )

    print()
    print("Expected Payroll:")

    for key, value in verification_result[
        "expected"
    ].items():
        print(
            f"{key}: {value}"
        )

    print()
    print("Differences:")

    for key, value in verification_result[
        "differences"
    ].items():
        print(
            f"{key}: {value}"
        )

    print()
    print("Overtime Violations:")

    if verification_result[
        "overtime_violations"
    ]:
        for violation in verification_result[
            "overtime_violations"
        ]:
            print(violation)
    else:
        print("None")

    if not verification_result[
        "discrepancies"
    ]:
        print()
        print("Payroll Update")
        print("--------------")
        print(
            "No update required. "
            "Payroll already matches the expected values."
        )
        return

    print()
    print(
        "Human approval is required before payroll update."
    )

    approval = input(
        "Approve payroll update? (yes/no): "
    ).strip().lower()

    if approval == "yes":
        update_result = update_payroll(
            employee_id,
            approved=True
        )
    else:
        update_result = update_payroll(
            employee_id,
            approved=False
        )

    print()
    print("Payroll Update")
    print("--------------")
    print(update_result)


print("Agent verification workflow is ready.")

In [ ]:
verify_and_approve("EMP001")